[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/docs/langchain-retrieval-augmentation.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/pinecone-io/examples/blob/master/docs/langchain-retrieval-augmentation.ipynb)

# Retrieval-Augmented Generation with Pinecone, LangChain and OpenAI

## Fixing LLMs that Hallucinate

In this notebook, you'll learn one of the most common applications of retrieval-augmented generation: giving large language models access to up-to-date information.


### Demo Data: Pinecone Documentation

A great example to use RAG is when augmenting LLMs with information that may not exist in their training data. This could private data, internal company information, or data that has been updated post a training cutoff. In our case, many modern LLMs are rained on Pinecone data that has since been updated, such as release notes, quickstart guides, blog posts, docs, and code.

In this example, we'll show the differences in generation from OpenAI's LLMs when asked about Pinecone's release notes! We'll orchestrate our RAG workflow using LangChain, a popular framework for AI applications.

In [1]:
!pip install -qU "langchain[openai]==1.2.7" langchain-text-splitters==1.1.0 langchain-pinecone==0.2.13 pinecone==7.3.0 pinecone-notebooks==0.1.1 requests==2.32.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
goo

In [2]:
try:
    import os
    from getpass import getpass

    import requests
    from langchain.chat_models import init_chat_model
    from langchain_core.documents import Document
    from langchain_openai import OpenAIEmbeddings
    from langchain_pinecone import PineconeVectorStore
    from langchain_text_splitters import MarkdownHeaderTextSplitter
    from pinecone import Pinecone, ServerlessSpec
except ImportError as e:
    raise ImportError(f"Missing required package. Run the dependency cell first: {e}")

---

## Building a Knowledge Base with our Vector Database

Building more reliable LLMs tools requires an external _"Knowledge Base"_, a database that we can query and update periodically with information.

Specifically, we will need to retrieve information that is relevant to our queries. To do this we need to use _"dense vector embeddings"_. These can be thought of as numerical representations of the *meaning* behind our sentences.

There are many options for creating these dense vectors, like open source [sentence transformers embedding models](https://www.pinecone.io/learn/series/nlp/) or OpenAI's [text-embedding-3-small model](https://platform.openai.com/docs/models/text-embedding-3-small). We will use OpenAI's offering in this example.

### Before you begin...

Be sure to grab a [free Pinecone account](https://app.pinecone.io/?sessionType=signup) and an OpenAI API key, [located here](https://platform.openai.com/api-keys)!

In [3]:
## Getting our Dataset:

# These are markdown versions of our release notes from 2025 and 2024
release_notes_2025 = "https://docs.pinecone.io/release-notes/2025.md"
release_notes_2024 = "https://docs.pinecone.io/release-notes/2024.md"

## Preprocessing our data

We'll use Requests and LangChain to pull down the release notes, and process the associated Markdown. We've used splitters that correspond to the release year, month/year, and features.
This will take us from raw Markdown files to LangChain Documents, which we'll embed and store in Pinecone.

In [12]:
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "release"), ("##", "month_year"), ("###", "feature")]
)


def download_link(url):
    response = requests.get(url)
    response.raise_for_status()
    return response.text


def add_document_metadata(doc, new_metadata):
    # returns new documents with updated metadata
    old_metadata = doc.metadata
    new_metadata = {**old_metadata, **new_metadata}
    return Document(page_content=doc.page_content, metadata=new_metadata)


def preprocess_pinecone_docs(urls):
    pinecone_docs = []
    for url in urls:
        # download the markdown
        response = download_link(url)
        split_text = splitter.split_text(response)
        # Update metadata to include url as source
        split_text = [
            add_document_metadata(doc, {"source": url, "chunk_num": num})
            for num, doc in enumerate(split_text)
        ]
        pinecone_docs.extend(split_text)
    return pinecone_docs


pinecone_docs = preprocess_pinecone_docs([release_notes_2024, release_notes_2025])

Let's take a closer look at one of these notes

In [5]:
print("Document content: ", pinecone_docs[2].page_content)
print("Document metadata: ", pinecone_docs[2].metadata)

Document content:  <Update label="2024-12-23" tags={["Database"]}>
Document metadata:  {'release': '2024 releases', 'month_year': 'December 2024', 'source': 'https://docs.pinecone.io/release-notes/2024.md', 'chunk_num': 2}


In [6]:
print("Document content: ", pinecone_docs[-1].page_content)
print("Document metadata: ", pinecone_docs[-1].metadata)

Document content:  Released [`v2.2.0`](https://github.com/pinecone-io/go-pinecone/releases/tag/v2.2.0) of the [Pinecone Go SDK](/reference/sdks/go/overview). This version adds support for [index tags](/guides/manage-data/manage-indexes#configure-index-tags) when creating or configuring indexes.
</Update>
Document metadata:  {'release': '2025 releases', 'month_year': 'January 2025', 'feature': 'Released Go SDK v2.2.0', 'source': 'https://docs.pinecone.io/release-notes/2025.md', 'chunk_num': 91}


## Setting up Pinecone


Next, we'll setup our API keys. For notebooks in Colab environments, we've included a handy block that helps set a Pinecone API key in your environment. In all other contexts, it's sufficient to save your Pinecone and OpenAI keys in your local environment.

Run the next two blocks and enter your Pinecone and OpenAI keys as needed:

In [7]:
def get_pinecone_api_key():
    """
    Get Pinecone API key from environment variable or prompt user for input.
    Returns the API key as a string.

    Only necessary for notebooks. When using Pinecone yourself,
    you can use environment variables or the like to set your API key.
    """
    api_key = os.environ.get("PINECONE_API_KEY")

    if api_key is None:
        try:
            # Try Colab authentication if available
            from pinecone_notebooks.colab import Authenticate

            Authenticate()
            # If successful, key will now be in environment
            api_key = os.environ.get("PINECONE_API_KEY")
        except ImportError:
            # If not in Colab or authentication fails, prompt user for API key
            print("Pinecone API key not found in environment.")
            api_key = getpass("Please enter your Pinecone API key: ")
            # Save to environment for future use in session
            os.environ["PINECONE_API_KEY"] = api_key

    return api_key


PINECONE_API_KEY = get_pinecone_api_key()

## Setup OpenAI API Key



In [13]:
def get_openai_api_key():
    """
    Get OpenAI API key from environment variable or prompt user for input.
    Returns the API key as a string.
    """

    api_key = os.environ.get("OPENAI_API_KEY")

    if api_key is None:
        try:
            api_key = getpass("Please enter your OpenAI API key: ")
            # Save to environment for future use in session
            os.environ["OPENAI_API_KEY"] = api_key
        except Exception as e:
            print(f"Error getting OpenAI API key: {e}")
            return None

    return api_key

In [9]:
OPENAI_API_KEY = get_openai_api_key()

Please enter your OpenAI API key: ··········


## Setting up Pinecone Index
We'll instantiate a Pinecone client, and create an index with a few key properties:
- index_name, which identifies our index
- dimension, which corresponds to the OpenAI embedding model vector size we'll use
- metric, which corresponds to the way "closeness" is evaluated with our vectors
- a spec, which determines the kind of index we are setting up. In this case, it's a free tier Pinecone serverless index

In [14]:
pc = Pinecone(
    api_key=PINECONE_API_KEY,
    source_tag="pinecone_examples:docs:langchain_retrieval_augmentation",
)

In [20]:
index_name = "langchain-pinecone-rag"

# 删除旧索引并创建一个支持 Pinecone 托管模型的索引
# multilingual-e5-large 模型需要 1024 维度
if pc.has_index(index_name):
    pc.delete_index(index_name)

pc.create_index(
    name=index_name,
    dimension=1024,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

index = pc.Index(name=index_name)
display(index.describe_index_stats())

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

## Embedding our documents and upserting into Pinecone

Next, we'll setup our OpenAI embedding model and Pinecone vector database within LangChain. To do this, we import the related abstractions from LangChain and pass in our API keys and model names.

After that, we generate ids for each document-chunk to better manage them within Pinecone.

In [15]:
[doc.metadata for doc in pinecone_docs]

[{'source': 'https://docs.pinecone.io/release-notes/2024.md', 'chunk_num': 0},
 {'release': '2024 releases',
  'source': 'https://docs.pinecone.io/release-notes/2024.md',
  'chunk_num': 1},
 {'release': '2024 releases',
  'month_year': 'December 2024',
  'source': 'https://docs.pinecone.io/release-notes/2024.md',
  'chunk_num': 2},
 {'release': '2024 releases',
  'month_year': 'December 2024',
  'feature': 'Increased namespaces limit',
  'source': 'https://docs.pinecone.io/release-notes/2024.md',
  'chunk_num': 3},
 {'release': '2024 releases',
  'month_year': 'December 2024',
  'feature': 'Pinecone Assistant JSON mode and EU region deployment',
  'source': 'https://docs.pinecone.io/release-notes/2024.md',
  'chunk_num': 4},
 {'release': '2024 releases',
  'month_year': 'December 2024',
  'feature': 'Released Spark-Pinecone connector v1.2.0',
  'source': 'https://docs.pinecone.io/release-notes/2024.md',
  'chunk_num': 5},
 {'release': '2024 releases',
  'month_year': 'December 2024',
 

In [23]:
from langchain_pinecone import PineconeVectorStore, PineconeEmbeddings

# Use PineconeEmbeddings to wrap the hosted model
# This keeps vectorization on the server side while providing the correct interface
embeddings = PineconeEmbeddings(model="multilingual-e5-large")

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

def clean_url_for_title(url):
    return url.split("/")[-1].replace(".md", "")

def generate_ids(doc_chunk):
    title = clean_url_for_title(doc_chunk.metadata["source"])
    chunk_num = doc_chunk.metadata["chunk_num"]
    feature = doc_chunk.metadata["feature"] if "feature" in doc_chunk.metadata else "na"
    return f"release_{title}#feature_{feature}#chunk_num{chunk_num}"

ids = [generate_ids(doc) for doc in pinecone_docs]

# Execute upload using server-side embeddings
vector_store.add_documents(documents=pinecone_docs, ids=ids)
print("数据已成功上传至 Pinecone！")

数据已成功上传至 Pinecone！


## Bringing it all together: Using OpenAI to learn about our releases

Finally, we'll setup an OpenAI LLM endpoint to generate responses given a user's query about our release notes!

In [25]:
# 已根据您的配置设置 base_url
llm = init_chat_model(
    "gpt-5.4-mini",
    model_provider="openai",
    api_key=OPENAI_API_KEY,
    base_url="https://api.llmhubapp.com/v1"
)

Next, let's run a query and retrieve some documents. These will be what is ultimately passed to our LLM that uses Pinecone to answer queries.

In [26]:
# OpenAI models will be unable to answer this due to training cutoffs in 2023-2024

query = "Tell me about version 7.0 of the Pinecone Python SDK"

retrieved_docs = vector_store.similarity_search(query, k=5)
docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

We can peek at the retrieved documents to confirm up to date information is being passed in:

In [27]:
for num, d in enumerate(retrieved_docs):
    print(f"Doc number: {num + 1}")
    print(d.page_content)
    print("Metadata:")
    print(d.metadata)
    print("-" * 100)

Doc number: 1
Released [`v7.1.0`](https://github.com/pinecone-io/pinecone-python-client/releases/tag/v7.1.0), [`v7.2.0`](https://github.com/pinecone-io/pinecone-python-client/releases/tag/v7.2.0), and [`v7.3.0`](https://github.com/pinecone-io/pinecone-python-client/releases/tag/v7.3.0) of the [Pinecone Python SDK](/reference/sdks/python/overview).  
* `v7.1.0` fixes minor bugs.
* `v7.2.0` adds support for [managing namespaces](/guides/manage-data/manage-namespaces).
* `v7.3.0` adds support for admin API operations for working with API keys, projects, and service accounts.
</Update>  
<Update label="2025-06-16" tags={["SDK"]}>
Metadata:
{'chunk_num': 41.0, 'feature': 'Released Python SDK v7.1.0, v7.2.0, and v7.3.0', 'month_year': 'June 2025', 'release': '2025 releases', 'source': 'https://docs.pinecone.io/release-notes/2025.md'}
----------------------------------------------------------------------------------------------------
Doc number: 2
Released [`v7.0.1`](https://github.com/pineco

### Comparing responses with/without RAG

Without our RAG pipeline with release notes indexed, OpenAI models will be
unable to answer questions about new versions of Pinecone. They may even "hallucinate", or fabricate information about the versions that may not exist!

In [28]:
print(llm.invoke(query).content)

Pinecone Python SDK **7.0** is a major version that introduced some fairly significant API and behavior changes compared with older releases.

## High-level idea
Version 7.0 is part of Pinecone’s move toward a more explicit, typed, and resource-oriented client design. In practice, that means:

- clearer client initialization
- more structured namespace/index operations
- better alignment with newer Pinecone service APIs
- breaking changes from older `pinecone-client` style code

## What changed in 7.0
Some of the most notable shifts in the 7.x line, including 7.0, are:

### 1. Newer client construction patterns
Older examples often used imports and setup styles that no longer apply. In 7.0, you generally use the modern `Pinecone` client object rather than legacy module-level functions.

Typical pattern:

```python
from pinecone import Pinecone

pc = Pinecone(api_key="YOUR_API_KEY")
index = pc.Index("my-index")
```

### 2. More explicit index management
Index creation, listing, and dele

However, with our pipeline in place, we get an answer that is more likely to be correct, and definitely grounded.

In [29]:
prompt = f"""You are an assistant that answers questions exclusively about the
Pinecone SDK release notes:

Here's a question: {query}

Here's some context from the release notes:

{docs_content}

Answer:
"""

# This will take a few seconds to run, due to the generation of the response from OpenAI
answer = llm.invoke(prompt)

print(answer.content)

Version 7.0 of the Pinecone Python SDK was released as **v7.0.0** on **2025-05-29**.

### What’s new in v7.0.0
It switched to the latest stable Pinecone API version, **`2025-04`**, and added support for:

- **Creating and managing backups**
- **Restoring indexes from backups**
- **Listing embedding and reranking models hosted by Pinecone**
- **Getting details about a model hosted by Pinecone**
- **Creating a BYOC (bring your own cloud) index**

### Other notable change
- The **`pinecone-plugin-assistant`** package is now **included by default**, so you no longer need to install it separately to use **Pinecone Assistant**.

### Follow-up patch releases
After `v7.0.0`, Pinecone released:
- **v7.0.1**
- **v7.0.2**

These were **minor bugfix releases** for issues discovered after the major version release.


## Wrapping it all in a function

In [30]:
def generate_response(query, use_pinecone=False):
    # Function to easily generate a response with and without Pinecone data

    if use_pinecone:
        retrieved_docs = vector_store.similarity_search(query, k=5)
        docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
        prompt = f"""
        You are an assistant that answers questions exclusively about the
        Pinecone SDK release notes:

        Here's a question: {query}

        Here's some context from the release notes:

        {docs_content}

        Answer: """

        # print out retrieved documents
        print("Retrieved documents:.....")
        for num, d in enumerate(retrieved_docs):
            print(f"Doc number: {num + 1}")
            print(d.page_content)
            print("Metadata:")
            print(d.metadata)
            print("-" * 100)
        print("Chatbot response:.....")

        return llm.invoke(prompt).content
    else:
        # no context is passed
        return llm.invoke(query).content

In [31]:
query = (
    "Tell me about recent changes to the pinecone-sparse embedding model context window"
)

print(generate_response(query, use_pinecone=False))

I don’t have live release notes, so I can’t reliably tell you the *most recent* changes to Pinecone’s `pinecone-sparse` embedding model context window without risking being outdated.

What I can tell you:

- The **context window** for an embedding model is the max amount of text it can take in one request.
- For **sparse embedding models**, this often matters less than for dense LLM-style models, but there can still be limits on:
  - total input characters/tokens
  - number of terms
  - per-item payload size
- If Pinecone changed it recently, it would typically be mentioned in one of:
  - Pinecone docs / model card
  - API changelog
  - release notes
  - SDK changelog if the limit is exposed there

If you want, I can help you quickly verify it by:
1. looking at the specific Pinecone docs page you’re using, or
2. checking an error message / model metadata if you paste it here.

If you have a link to the docs or release note, send it over and I’ll summarize the change for you.


In [32]:
print(generate_response(query, use_pinecone=True))

Retrieved documents:.....
Doc number: 1
You can now raise the context window for Pinecone's hosted [`pinecone-sparse-english-v0`](/guides/index-data/create-an-index#pinecone-sparse-english-v0) embedding model from `512` to `2048` using the `max_tokens_per_sequence` parameter.
</Update>  
<Update label="2025-07-23" tags={["SDK"]}>
Metadata:
{'chunk_num': 35.0, 'feature': 'Increased context window for `pinecone-sparse-english-v0`', 'month_year': 'July 2025', 'release': '2025 releases', 'source': 'https://docs.pinecone.io/release-notes/2025.md'}
----------------------------------------------------------------------------------------------------
Doc number: 2
Released [`v3.0.0`](https://github.com/pinecone-io/pinecone-dotnet-client/releases/tag/3.0.0) of the [Pinecone .NET SDK](/reference/sdks/dotnet/overview). This version uses the latest stable API version, `2025-01`, and adds support for [sparse-only indexes](/guides/index-data/indexing-overview#indexes-with-sparse-vectors).  
<Warning>

## Wrapping up

And that's that! You've successfully implemented retrieval augmented generation with Pinecone, OpenAI and LangChain. Wanna learn more? Try implementing the following:

- Sparse search to enable precise time, date and feature recognition in query results
- Expanding the set of documents to encompass all Pinecone documentaiotn
- Learning how to chunk and process code data, to build your own code assistant

To finish, let's delete our index:

In [33]:
pc.delete_index(name=index_name)

---